# Queuechella — Music Festival Simulation

**Simulation Course, Semester B 2026 — Ben-Gurion University**

A discrete-event simulation (DES) of a two-day music festival. The model includes three visitor types (`FriendsGroup`, `Couple`, `Single`), six service stations (FIFO queues with multiple servers), and three capacity-limited stages.

**Goal**: compare three upgrade alternatives against a baseline scenario under a 1,000,000 NIS budget cap.

The notebook layout mirrors the example HotelSimulation project:
1. Goodness-of-fit on Excel data
2. Heating-time analysis (Welch's moving average)
3. Baseline scenario — Data Points For Current State
4. Alternative Review
5. Comparison of Alternatives By Welch
6. Conclusions and recommendations

## Environment setup

The simulation modules (`engine`, `sim_stats`, `distributions`, etc.) are already written as `.py` files in the project. The notebook imports them directly — this works both in VS Code (when the notebook sits in the project folder) and in Colab once the repository has been cloned.

In [ ]:
# When running on Colab: clone the repo and switch into the project folder
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.isdir('simu_v2'):
        !git clone https://github.com/saaragmon/simu_v2.git
    os.chdir('simu_v2')
    !pip install -q scipy openpyxl matplotlib

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
# imports
import time
import math
import statistics as stats_lib
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

from config import SimConfig
from engine import Simulation
from sim_stats import MultiRunStatistics, RunStatistics
from alternatives import build_baseline, build_combo_a, build_combo_b, build_combo_c
from distributions import fit_from_excel
from warmup import WarmupSimulation

# Global parameters (kept in sync with main.py)
EXCEL_PATH = 'samples_for_simulation.xlsx'
NUM_RUNS = 20
CONFIDENCE_LEVEL = 0.90
RELATIVE_PRECISION = 0.10

KPIS = ['avg_satisfaction', 'total_revenue', 'avg_queue_length']
KPI_HIGHER_IS_BETTER = {
    'avg_satisfaction':   True,
    'total_revenue':      True,
    'avg_queue_length':   False,
}

## 1. Goodness-of-Fit on Excel data

The file `samples_for_simulation.xlsx` contains two columns:
- **Sheet 1**: inter-arrival times of `FriendsGroup` visitors
- **Sheet 2**: MainStage show durations

For each column we fit three candidate families (Exponential, Normal, Uniform) by MLE and select the distribution with the **smallest KS statistic** (Kolmogorov-Smirnov test). The winner is used as the sampler for that phenomenon throughout the simulation.

In [ ]:
samplers = fit_from_excel(EXCEL_PATH)
friends_sampler    = samplers['friends_interarrival']
main_stage_sampler = samplers['main_stage_duration']
print("\n-> Fitted distributions loaded as samplers.")

## 2. Heating-time analysis

We run 30 replications and inspect the per-day KPI behaviour using **Welch's moving average**. The goal is to identify how many simulated days the system needs to reach steady state — this initial transient is then discarded from the main statistical analysis.

In [ ]:
warmup = WarmupSimulation(n_runs=30)
warmup.run()
warmup.plot_heating_time_data(warmup.daily_avg_queue_lengths, 'Average Queue Length')
warmup.plot_heating_time_data(warmup.daily_avg_satisfactions, 'Average Satisfaction')

## 3. Data Points For Current State — Baseline scenario

Run the Baseline scenario for 20 independent replications. Each replication uses a different seed (1000, 1001, ..., 1019) so the runs are independent yet reproducible.

In [ ]:
def run_scenario(name, cfg, num_runs, friends_sampler, main_stage_sampler, base_seed):
    """Run `num_runs` replications of a scenario and return MultiRunStatistics."""
    multi = MultiRunStatistics(CONFIDENCE_LEVEL, RELATIVE_PRECISION)
    print(f"\n  Running '{name}' — {num_runs} replications ...")
    for i in range(num_runs):
        sim = Simulation(cfg,
                         friends_arrival_sampler=friends_sampler,
                         main_stage_duration_sampler=main_stage_sampler)
        s = sim.run(seed=base_seed + i)
        multi.add_run(s)
    print(f"  Done.")
    return multi

baseline_alt   = build_baseline()
baseline_stats = run_scenario('Baseline', baseline_alt.config, NUM_RUNS,
                              friends_sampler, main_stage_sampler, base_seed=1000)
print(baseline_stats.report())

### Required replications (relative precision check)

For each KPI we test whether 20 replications are enough to reach relative precision γ = 0.1 at confidence level 0.9. The criterion is:

$$\frac{\delta(\alpha,n)}{|\bar{x}|} \le \frac{\gamma}{1+\gamma}$$

where $\delta(\alpha,n) = t_{n-1,1-\alpha/2} \cdot s/\sqrt{n}$. If the criterion is not met, we also compute how many additional replications would be needed.

In [ ]:
def relative_accuracy_check(multi, kpi, gamma=0.10):
    """Check whether the current replication count meets relative precision `gamma`.

    Returns a dict with the mean, std, relative error, threshold gamma/(1+gamma),
    a boolean for whether the criterion is met, and the additional replication
    count needed if not.
    """
    values = multi._kpi_values(kpi)
    n = len(values)
    mean = stats_lib.mean(values)
    std  = stats_lib.stdev(values)
    alpha = 1.0 - multi.confidence_level
    t_crit = student_t.ppf(1 - alpha / 2, df=n-1)
    delta = t_crit * std / math.sqrt(n)
    rel_err   = delta / abs(mean) if mean else float('inf')
    threshold = gamma / (1 + gamma)
    meets     = rel_err <= threshold
    n_needed  = multi.required_replications(kpi, pilot_runs=n)
    return {
        'kpi': kpi, 'mean': mean, 'std': std, 'n': n,
        'relative_error': rel_err, 'threshold': threshold,
        'meets_criterion': meets, 'n_needed': n_needed,
    }

print(f"{'KPI':25s} {'mean':>14s} {'rel_err':>10s} {'γ/(1+γ)':>12s} {'meets?':>8s} {'n needed':>10s}")
print('-' * 82)
for kpi in KPIS:
    r = relative_accuracy_check(baseline_stats, kpi, gamma=RELATIVE_PRECISION)
    print(f"{r['kpi']:25s} {r['mean']:14.4f} {r['relative_error']:10.4f} "
          f"{r['threshold']:12.4f} {str(r['meets_criterion']):>8s} {r['n_needed']:>10d}")

### Confidence intervals for Baseline KPIs

Student-t confidence intervals for each KPI at confidence level 90%:

$$\bar{x}(n) \pm t_{n-1,1-\alpha/2} \cdot \frac{s(n)}{\sqrt{n}}$$

In [ ]:
print(f"{'KPI':25s} {'mean':>14s} {'CI lower':>14s} {'CI upper':>14s}")
print('-' * 70)
for kpi in KPIS:
    mean, lo, hi = baseline_stats.confidence_interval(kpi)
    print(f"{kpi:25s} {mean:14.4f} {lo:14.4f} {hi:14.4f}")

## 4. Alternative Review

Three upgrade alternatives, all within the 1,000,000 NIS budget cap. Each alternative is run separately with a distinct base seed (Combo_A: 11000+, Combo_B: 21000+, Combo_C: 31000+) — so **the replications across alternatives are independent**, which is what Welch's t-test requires.

**The three alternatives**:
- **Combo_A** (950,000 NIS): Extra photo + body-art + Marketing + Auto ticket scanning — *Overall Winner* (lowest rank-sum across all three KPIs)
- **Combo_B** (1,000,000 NIS): Marketing + Auto ticket scanning + Visitor gift bag — *Revenue King* (highest total revenue)
- **Combo_C** (650,000 NIS): Popular bands + Extra photo + body-art + Visitor gift bag — *Satisfaction King* (highest avg satisfaction)

In [ ]:
combo_a_alt   = build_combo_a()
combo_a_stats = run_scenario(combo_a_alt.name, combo_a_alt.config, NUM_RUNS,
                             friends_sampler, main_stage_sampler, base_seed=11000)
print(combo_a_stats.report())

In [ ]:
combo_b_alt   = build_combo_b()
combo_b_stats = run_scenario(combo_b_alt.name, combo_b_alt.config, NUM_RUNS,
                             friends_sampler, main_stage_sampler, base_seed=21000)
print(combo_b_stats.report())

In [ ]:
combo_c_alt   = build_combo_c()
combo_c_stats = run_scenario(combo_c_alt.name, combo_c_alt.config, NUM_RUNS,
                             friends_sampler, main_stage_sampler, base_seed=31000)
print(combo_c_stats.report())

## 5. Comparison of Alternatives By Welch

**Welch's t-test assumptions:**
- No dependence between simulation runs executed in parallel.
- No dependence between the runs within each alternative.
- We cannot assume that the variances across alternatives are equal.
- The KPIs being tested are normally distributed.

*The normality assumption holds because the KPIs are averages, so the Central Limit Theorem applies. Independence across alternatives holds because we used `random.random` with a different seed range per alternative (Baseline: 1000+, Combo_A: 11000+, Combo_B: 21000+, Combo_C: 31000+).*

**Test formula** — build a confidence interval on the difference between the two scenario means:

$$\bar{x}_1(n_1) - \bar{x}_2(n_2) \pm t_{\hat{f},1-\alpha/2} \cdot \sqrt{\frac{s_1^2(n_1)}{n_1} + \frac{s_2^2(n_2)}{n_2}}$$

where $\hat{f}$ is set by the Welch-Satterthwaite equation:

$$\hat{f} = \frac{\left[\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right]^2}{\frac{(s_1^2/n_1)^2}{n_1-1} + \frac{(s_2^2/n_2)^2}{n_2-1}}$$

**Interpretation**: if 0 falls inside the confidence interval, we cannot determine which alternative is better. If the interval is entirely positive or entirely negative, we conclude in favour of the corresponding side.

In [ ]:
def print_welch_comparison(baseline, alternative, alt_name):
    """Print the Welch CI on the mean difference (baseline - alternative) for each KPI."""
    conf_pct = int(round(CONFIDENCE_LEVEL * 100))
    print(f"\n  --- {alt_name} vs Baseline (Welch's t-test) ---")
    for kpi in KPIS:
        r = baseline.welch_t_test(alternative, kpi)
        diff, lo, hi = r['diff'], r['ci_lower'], r['ci_upper']
        higher_is_better = KPI_HIGHER_IS_BETTER.get(kpi, True)
        print(f"    {kpi}")
        print(f"      Mean difference (baseline - {alt_name}): {diff:+.4f}")
        print(f"      {conf_pct}% Confidence Interval: [{lo:+.4f}, {hi:+.4f}]")
        if lo > 0:
            winner = 'Baseline' if higher_is_better else alt_name
            print(f"      0 outside CI -> {winner} significantly better")
        elif hi < 0:
            winner = alt_name if higher_is_better else 'Baseline'
            print(f"      0 outside CI -> {winner} significantly better")
        else:
            print("      0 inside CI -> cannot determine which is better")

print_welch_comparison(baseline_stats, combo_a_stats, 'Combo_A')
print_welch_comparison(baseline_stats, combo_b_stats, 'Combo_B')
print_welch_comparison(baseline_stats, combo_c_stats, 'Combo_C')

## 6. Project Analysis — Conclusions and Recommendations

The project uses event-driven simulation to model the day-to-day operation of a two-day music festival, tracking three top-level KPIs — **visitor satisfaction**, **total revenue**, and **average queue length**.

Within the 1,000,000 NIS budget, we evaluated three alternatives:

1. **Combo_A** (950,000 NIS) — focuses on removing the entry bottleneck and shortening service queues.
2. **Combo_B** (1,000,000 NIS) — focuses on commercial ROI (marketing + entry efficiency + initial satisfaction).
3. **Combo_C** (650,000 NIS) — the cheapest; focuses on satisfaction lift (popular bands + gift bag).

### Results of the Welch comparison

*(Fill in based on the actual run output — for each KPI: was 0 inside the CI? If outside, which side?)*

**Combo_A**: ...

**Combo_B**: ...

**Combo_C**: ...

### Final recommendation

...